In [1]:
import requests
import time
import pandas as pd
from collections import deque

API_KEY = "9706027F573365120F82AC2AA09940A0"  # <-- replace with your Steam Web API key
BASE_URL = "https://api.steampowered.com"
RATE_LIMIT = 1.5
MAX_USERS = 100

# Seed users (5 public popular SteamIDs)
seed_users = [
    "76561198181304338",
    "76561197960279927",
    "76561197960461627",
    "76561198284975110",
    "76561198230355712"
]

queue = deque(seed_users)
visited = set()

users_rows = []
games_rows = []
friends_rows = []

# Helper functions
def get_owned_games(steamid):
    url = f"{BASE_URL}/IPlayerService/GetOwnedGames/v1/"
    params = {"key": API_KEY, "steamid": steamid, "include_appinfo": True}
    r = requests.get(url, params=params)
    if r.status_code != 200:
        return None
    return r.json().get("response", {}).get("games")

def get_friends(steamid):
    url = f"{BASE_URL}/ISteamUser/GetFriendList/v1/"
    params = {"key": API_KEY, "steamid": steamid}
    r = requests.get(url, params=params)
    if r.status_code != 200:
        return []
    return [f["steamid"] for f in r.json().get("friendslist", {}).get("friends", [])]

def get_user_metadata(steamids):
    url = f"{BASE_URL}/ISteamUser/GetPlayerSummaries/v2/"
    params = {"key": API_KEY, "steamids": ",".join(steamids)}
    r = requests.get(url, params=params)
    if r.status_code != 200:
        return {}
    return {p["steamid"]: p for p in r.json().get("response", {}).get("players", [])}

# Crawl loop
while queue and len(visited) < MAX_USERS:
    steamid = queue.popleft()
    if steamid in visited:
        continue

    print(f"Processing user {len(visited)+1}: {steamid}")
    visited.add(steamid)

    # Get metadata
    metadata = get_user_metadata([steamid]).get(steamid, {})
    country = metadata.get("loccountrycode")
    timecreated = metadata.get("timecreated")

    # Get owned games
    games = get_owned_games(steamid)
    time.sleep(RATE_LIMIT)
    if games is None:
        continue  # private profile, skip

    total_playtime = sum(g["playtime_forever"] for g in games)

    users_rows.append({
        "steamid": steamid,
        "total_games": len(games),
        "total_playtime": total_playtime,
        "country": country,
        "account_created": timecreated
    })

    for g in games:
        games_rows.append({
            "steamid": steamid,
            "appid": g["appid"],
            "game_name": g["name"],
            "playtime_minutes": g["playtime_forever"]
        })

    # Get friends
    friends = get_friends(steamid)
    time.sleep(RATE_LIMIT)

    for f in friends:
        friends_rows.append({"user_steamid": steamid, "friend_steamid": f})
        if f not in visited:
            queue.append(f)

# Save CSVs
pd.DataFrame(users_rows).to_csv("users.csv", index=False)
pd.DataFrame(games_rows).to_csv("games.csv", index=False)
pd.DataFrame(friends_rows).to_csv("friends.csv", index=False)

print("Done! CSV files saved.")

Processing user 1: 76561198181304338
Processing user 2: 76561197960279927
Processing user 3: 76561197960461627
Processing user 4: 76561198284975110
Processing user 5: 76561198230355712
Processing user 6: 76561198043335723
Processing user 7: 76561198051819891
Processing user 8: 76561198064046042
Processing user 9: 76561198093456007
Processing user 10: 76561198098783505
Processing user 11: 76561198103514380
Processing user 12: 76561198104073351
Processing user 13: 76561198113109970
Processing user 14: 76561198120723341
Processing user 15: 76561198133604673
Processing user 16: 76561198137641413
Processing user 17: 76561198138920519
Processing user 18: 76561198140312583
Processing user 19: 76561198142624795
Processing user 20: 76561198151352759
Processing user 21: 76561198153416192
Processing user 22: 76561198157436029
Processing user 23: 76561198157745879
Processing user 24: 76561198195166719
Processing user 25: 76561198196191411
Processing user 26: 76561198202793371
Processing user 27: 7